In [1]:

from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

# Find the repository root
root = Path.cwd()
if not (root / "data/processed").exists():
    root = root.parent

data_path = root / "data/processed/patch_model_eligible_terrain_v2.parquet"
roles_path = root / "data/processed/patch_model_eligible_terrain_v2.roles.json"

assert data_path.exists(), "Dataset not found"
assert roles_path.exists(), "Feature roles not found"

# Load column-role definitions
with open(roles_path, "r") as f:
    roles = json.load(f)

# Inspect the dataset without reading any target values
parquet = pq.ParquetFile(data_path)

print("Dataset rows:", parquet.metadata.num_rows)
print("Dataset columns:", len(parquet.schema_arrow.names))
print("\nRoles JSON structure:")
print(json.dumps(roles, indent=2)[:3000])


Dataset rows: 111257
Dataset columns: 129

Roles JSON structure:
{
  "target": "nbr_5_mean",
  "predictors": [
    "nbr_pre_mean",
    "dominant_evt",
    "dominant_evt_fraction",
    "tree_cover_mean",
    "tree_height_mean",
    "elevation_mean",
    "slope_mean",
    "northness_mean",
    "eastness_mean",
    "fire_size_acres",
    "ignition_month",
    "mtbs_unburned_low_fraction",
    "mtbs_low_fraction",
    "mtbs_moderate_fraction",
    "mtbs_high_fraction",
    "mtbs_increased_greenness_fraction",
    "hist_prcp_mm",
    "hist_tmax_c",
    "hist_tmin_c",
    "hist_s1_vpd_kpa",
    "nbr_1_mean",
    "pre_s1_prcp_mm",
    "pre_s1_tmax_c",
    "pre_s1_tmin_c",
    "pre_s1_vpd_kpa",
    "pre_s1_prcp_anom_mm",
    "pre_s1_tmax_anom_c",
    "pre_s1_tmin_anom_c",
    "pre_s1_vpd_anom_kpa",
    "s1_prcp_anom_mm",
    "s1_tmax_anom_c",
    "s1_tmin_anom_c",
    "s1_vpd_anom_kpa",
    "nbr_drop_patch"
  ],
  "early_predictors": [
    "nbr_pre_mean",
    "dominant_evt",
    "dominant_evt_

In [2]:

# Approved inputs and target
features = roles["predictors"]
target = roles["target"]

assert len(features) == 34
assert target not in features

keys = ["fire_id", "patch_id", "split"]

# Load approved inputs for all splits
inputs = pd.read_parquet(
    data_path,
    columns=keys + features
)

# Load targets ONLY for training and validation
dev_targets = pd.read_parquet(
    data_path,
    columns=keys + [target],
    filters=[("split", "in", ["train", "validation"])]
)

# Combine inputs and labels for development data
dev = inputs.loc[
    inputs["split"].isin(["train", "validation"])
].merge(
    dev_targets,
    on=keys,
    validate="one_to_one"
)

train = dev.loc[dev["split"] == "train"].copy()
val = dev.loc[dev["split"] == "validation"].copy()

# Separate predictors (X) and target (y)
X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]

# Safety checks
assert len(train) == 61090
assert len(val) == 19561
assert inputs["split"].eq("test").sum() == 30606
assert train["fire_id"].nunique() == 78
assert val["fire_id"].nunique() == 36
assert set(train["fire_id"]).isdisjoint(val["fire_id"])
assert y_train.notna().all() and y_val.notna().all()

print("Training:", X_train.shape, "| Fires:", train["fire_id"].nunique())
print("Validation:", X_val.shape, "| Fires:", val["fire_id"].nunique())
print("Test inputs:", inputs["split"].eq("test").sum())
print("Target:", target)
print("Approved features:", len(features))


Training: (61090, 34) | Fires: 78
Validation: (19561, 34) | Fires: 36
Test inputs: 30606
Target: nbr_5_mean
Approved features: 34


In [3]:

import numpy as np
import sklearn
print(sklearn.__version__)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Learn the average NBR5 from TRAINING data only
train_mean = y_train.mean()

# 2. Predict that same value for every validation patch
mean_predictions = np.full(len(y_val), train_mean)

# 3. Calculate prediction errors
mae = mean_absolute_error(y_val, mean_predictions)
rmse = np.sqrt(mean_squared_error(y_val, mean_predictions))
r2 = r2_score(y_val, mean_predictions)

# 4. Give each validation wildfire equal importance
errors = pd.DataFrame({
    "fire_id": val["fire_id"].to_numpy(),
    "absolute_error": np.abs(y_val.to_numpy() - mean_predictions)
})

equal_fire_mae = (
    errors.groupby("fire_id")["absolute_error"]
    .mean()
    .mean()
)

print("MEAN BASELINE — VALIDATION RESULTS")
print(f"Training mean NBR5: {train_mean:.4f}")
print(f"Validation MAE:      {mae:.4f}")
print(f"Validation RMSE:     {rmse:.4f}")
print(f"Validation R²:       {r2:.4f}")
print(f"Equal-fire MAE:      {equal_fire_mae:.4f}")


1.9.1
MEAN BASELINE — VALIDATION RESULTS
Training mean NBR5: 0.3122
Validation MAE:      0.1624
Validation RMSE:     0.1932
Validation R²:       -0.0255
Equal-fire MAE:      0.1421


In [4]:

# Predict NBR5 using each patch's observed NBR1
persistence_predictions = X_val["nbr_1_mean"].to_numpy()

# Calculate validation prediction errors
persistence_mae = mean_absolute_error(
    y_val, persistence_predictions
)
persistence_rmse = np.sqrt(
    mean_squared_error(y_val, persistence_predictions)
)
persistence_r2 = r2_score(
    y_val, persistence_predictions
)

# Give each wildfire equal weight
persistence_errors = pd.DataFrame({
    "fire_id": val["fire_id"].to_numpy(),
    "absolute_error": np.abs(
        y_val.to_numpy() - persistence_predictions
    )
})

persistence_equal_fire_mae = (
    persistence_errors.groupby("fire_id")["absolute_error"]
    .mean()
    .mean()
)

print("PERSISTENCE BASELINE — VALIDATION RESULTS")
print(f"Validation MAE:     {persistence_mae:.4f}")
print(f"Validation RMSE:    {persistence_rmse:.4f}")
print(f"Validation R²:      {persistence_r2:.4f}")
print(f"Equal-fire MAE:     {persistence_equal_fire_mae:.4f}")

# Compare both baselines
comparison = pd.DataFrame({
    "Model": ["Training mean", "NBR1 persistence"],
    "MAE": [mae, persistence_mae],
    "RMSE": [rmse, persistence_rmse],
    "R²": [r2, persistence_r2],
    "Equal-fire MAE": [equal_fire_mae, persistence_equal_fire_mae]
})

print("\nBASELINE COMPARISON")
display(comparison.round(4))


PERSISTENCE BASELINE — VALIDATION RESULTS
Validation MAE:     0.1839
Validation RMSE:    0.2423
Validation R²:      -0.6139
Equal-fire MAE:     0.1585

BASELINE COMPARISON


,Model,MAE,RMSE,R²,Equal-fire MAE
0,Training mean,0.1624,0.1932,-0.0255,0.1421
1,NBR1 persistence,0.1839,0.2423,-0.6139,0.1585


In [5]:

import mlflow

# Store experiment records in a local SQLite database
db_path = root / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{db_path.as_posix()}"
)

mlflow.set_experiment("P1 - Wildfire Recovery - Phase 5")

print("MLflow version:", mlflow.__version__)
print("Tracking location:", mlflow.get_tracking_uri())
print("Experiment ready!")


MLflow version: 3.17.2
Tracking location: sqlite:///c:/F copy/wildfire-recovery-forecasting/mlflow.db
Experiment ready!


In [6]:

# Shared information about our experiments
experiment_info = {
    "dataset": "patch_model_eligible_terrain_v2",
    "train_years": "2015-2017",
    "validation_years": "2018-2019",
    "train_fires": 78,
    "validation_fires": 36,
}

# Experiment 1: Training-mean baseline
with mlflow.start_run(run_name="baseline_training_mean"):
    mlflow.log_params({
        **experiment_info,
        "strategy": "predict_training_mean",
        "constant_prediction": float(train_mean),
    })

    mlflow.log_metrics({
        "val_mae": float(mae),
        "val_rmse": float(rmse),
        "val_r2": float(r2),
        "val_equal_fire_mae": float(equal_fire_mae),
    })

print("Logged training-mean baseline!")

# Experiment 2: NBR1 persistence baseline
with mlflow.start_run(run_name="baseline_nbr1_persistence"):
    mlflow.log_params({
        **experiment_info,
        "strategy": "predict_nbr1",
        "predictor": "nbr_1_mean",
    })

    mlflow.log_metrics({
        "val_mae": float(persistence_mae),
        "val_rmse": float(persistence_rmse),
        "val_r2": float(persistence_r2),
        "val_equal_fire_mae": float(persistence_equal_fire_mae),
    })

print("Logged NBR1 persistence baseline!")

# Retrieve both experiments from MLflow
runs = mlflow.search_runs(
    experiment_names=["P1 - Wildfire Recovery - Phase 5"]
)

display(
    runs[
        [
            "tags.mlflow.runName",
            "metrics.val_mae",
            "metrics.val_rmse",
            "metrics.val_r2",
            "metrics.val_equal_fire_mae",
        ]
    ].round(4)
)


Logged training-mean baseline!
Logged NBR1 persistence baseline!


,tags.mlflow.runName,metrics.val_mae,metrics.val_rmse,metrics.val_r2,metrics.val_equal_fire_mae
0,baseline_nbr1_persistence,0.1839,0.2423,-0.6139,0.1585
1,baseline_training_mean,0.1624,0.1932,-0.0255,0.1421
2,baseline_nbr1_persistence,0.1839,0.2423,-0.6139,0.1585
3,baseline_training_mean,0.1624,0.1932,-0.0255,0.1421


In [7]:
from wildfire_recovery_forecasting.evaluation import evaluate_model

# Check that the function reproduces our original results
mean_scores = evaluate_model(
    y_val, mean_predictions, val["fire_id"]
)

persistence_scores = evaluate_model(
    y_val, persistence_predictions, val["fire_id"]
)

assert np.isclose(mean_scores["mae"], mae)
assert np.isclose(mean_scores["equal_fire_mae"], equal_fire_mae)
assert np.isclose(persistence_scores["mae"], persistence_mae)
assert np.isclose(
    persistence_scores["equal_fire_mae"],
    persistence_equal_fire_mae
)

display(pd.DataFrame({
    "Training mean": mean_scores,
    "NBR1 persistence": persistence_scores
}).T.round(4))

print("Reusable evaluation function verified!")


,mae,rmse,r2,equal_fire_mae
Training mean,0.1624,0.1932,-0.0255,0.1421
NBR1 persistence,0.1839,0.2423,-0.6139,0.1585


Reusable evaluation function verified!


# RIDGE REGRESSION

In [8]:

# Phase 5 Day 2 — Confirm Ridge modeling inputs

categorical_features = [
    "dominant_evt",
    "ignition_month",
]

# Drop only the exactly redundant NBR difference for Ridge
ridge_features = [
    f for f in features
    if f != "nbr_drop_patch"
]

numeric_features = [
    f for f in ridge_features
    if f not in categorical_features
]

assert len(features) == 34
assert len(ridge_features) == 33
assert len(numeric_features) == 31

assert len(X_train) == 61090
assert len(X_val) == 19561

print("Training patches:", len(X_train))
print("Validation patches:", len(X_val))
print("Ridge input features:", len(ridge_features))
print("Numerical features:", len(numeric_features))
print("Categorical features:", categorical_features)

print("\nMissing training feature values:")
print(X_train[ridge_features].isna().sum().sum())

print("\nCategorical feature types:")
print(X_train[categorical_features].dtypes)


Training patches: 61090
Validation patches: 19561
Ridge input features: 33
Numerical features: 31
Categorical features: ['dominant_evt', 'ignition_month']

Missing training feature values:
0

Categorical feature types:
dominant_evt      int64
ignition_month    int64
dtype: object


In [9]:
from wildfire_recovery_forecasting.modeling import build_ridge_pipeline

ridge_pipeline = build_ridge_pipeline(features, alpha=1.0)
preprocessor = ridge_pipeline.named_steps["preprocess"]

# 3. Check that validation inputs are also complete
assert X_val[ridge_features].notna().all().all()

print("Ridge pipeline created!")
print("\nPipeline steps:")
for name, component in ridge_pipeline.steps:
    print(f"  {name}: {component.__class__.__name__}")

print("\nNumerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Ready for training and cross-validation.")


Ridge pipeline created!

Pipeline steps:
  preprocess: ColumnTransformer
  model: Ridge

Numerical features: 31
Categorical features: 2
Ready for training and cross-validation.


In [10]:

from sklearn.model_selection import GroupKFold

# Each patch belongs to a wildfire
groups = train["fire_id"]

# Create three folds, keeping each fire together
cv = GroupKFold(n_splits=3)

cv_splits = list(
    cv.split(
        X_train[ridge_features],
        y_train,
        groups=groups
    )
)

summary = []

for fold, (train_idx, holdout_idx) in enumerate(cv_splits, 1):

    training_fires = set(groups.iloc[train_idx])
    holdout_fires = set(groups.iloc[holdout_idx])

    # A fire must never appear in both groups
    assert training_fires.isdisjoint(holdout_fires)

    summary.append({
        "Fold": fold,
        "Training patches": len(train_idx),
        "Held-out patches": len(holdout_idx),
        "Training fires": len(training_fires),
        "Held-out fires": len(holdout_fires),
        "Fire overlap": len(training_fires & holdout_fires)
    })

assert sum(len(v) for _, v in cv_splits) == len(train)

display(pd.DataFrame(summary))

print("3-fold fire-grouped CV ready!")


,Fold,Training patches,Held-out patches,Training fires,Held-out fires,Fire overlap
0,1,40727,20363,51,27,0
1,2,40726,20364,53,25,0
2,3,40727,20363,52,26,0


3-fold fire-grouped CV ready!


In [11]:

from sklearn.base import clone

# Four regularization strengths to compare
alphas = [0.1, 1.0, 10.0, 100.0]

cv_results = []

for alpha in alphas:
    fold_maes = []

    for train_idx, holdout_idx in cv_splits:

        # Create a fresh preprocessing + Ridge pipeline
        model = clone(ridge_pipeline)
        model.set_params(model__alpha=alpha)

        # Train on this fold's training fires only
        model.fit(
            X_train.iloc[train_idx][ridge_features],
            y_train.iloc[train_idx]
        )

        # Predict NBR5 for held-out training fires
        predictions = model.predict(
            X_train.iloc[holdout_idx][ridge_features]
        )

        fold_maes.append(
            mean_absolute_error(
                y_train.iloc[holdout_idx],
                predictions
            )
        )

    cv_results.append({
        "alpha": alpha,
        "fold_1_mae": fold_maes[0],
        "fold_2_mae": fold_maes[1],
        "fold_3_mae": fold_maes[2],
        "mean_cv_mae": np.mean(fold_maes),
        "std_cv_mae": np.std(fold_maes)
    })

cv_results_df = (
    pd.DataFrame(cv_results)
    .sort_values("mean_cv_mae")
)

display(cv_results_df.round(5))

best_alpha = float(cv_results_df.iloc[0]["alpha"])

print(f"Best alpha from grouped CV: {best_alpha}")


,alpha,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,std_cv_mae
3,100.0,0.07552,0.07420,0.08281,0.07751,0.00378
2,10.0,0.07727,0.07601,0.09038,0.08122,0.00650
1,1.0,0.07810,0.07782,0.09241,0.08278,0.00681
0,0.1,0.07824,0.07813,0.09269,0.08302,0.00684


Best alpha from grouped CV: 100.0


In [12]:

more_alphas = [300.0, 1000.0, 3000.0]
additional_results = []

for alpha in more_alphas:
    fold_maes = []

    for train_idx, holdout_idx in cv_splits:
        model = clone(ridge_pipeline)
        model.set_params(model__alpha=alpha)

        model.fit(
            X_train.iloc[train_idx][ridge_features],
            y_train.iloc[train_idx]
        )

        predictions = model.predict(
            X_train.iloc[holdout_idx][ridge_features]
        )

        fold_maes.append(
            mean_absolute_error(
                y_train.iloc[holdout_idx],
                predictions
            )
        )

    additional_results.append({
        "alpha": alpha,
        "fold_1_mae": fold_maes[0],
        "fold_2_mae": fold_maes[1],
        "fold_3_mae": fold_maes[2],
        "mean_cv_mae": np.mean(fold_maes),
        "std_cv_mae": np.std(fold_maes)
    })

# Compare all seven alpha settings
cv_results_df = (
    pd.concat(
        [cv_results_df, pd.DataFrame(additional_results)],
        ignore_index=True
    )
    .drop_duplicates(subset="alpha", keep="last")
    .sort_values("mean_cv_mae")
    .reset_index(drop=True)
)

display(cv_results_df.round(5))

best_alpha = float(cv_results_df.iloc[0]["alpha"])
print(f"Current best alpha: {best_alpha}")


,alpha,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,std_cv_mae
0,3000.0,0.07256,0.07983,0.07038,0.07425,0.00404
1,1000.0,0.07352,0.07794,0.07261,0.07469,0.00233
2,300.0,0.07475,0.07571,0.07756,0.07600,0.00117
3,100.0,0.07552,0.07420,0.08281,0.07751,0.00378
4,10.0,0.07727,0.07601,0.09038,0.08122,0.00650
5,1.0,0.07810,0.07782,0.09241,0.08278,0.00681
6,0.1,0.07824,0.07813,0.09269,0.08302,0.00684


Current best alpha: 3000.0


In [13]:

final_alphas = [10000.0, 30000.0, 100000.0]
final_results = []

for alpha in final_alphas:
    fold_maes = []

    for train_idx, holdout_idx in cv_splits:
        model = clone(ridge_pipeline)
        model.set_params(model__alpha=alpha)

        model.fit(
            X_train.iloc[train_idx][ridge_features],
            y_train.iloc[train_idx]
        )

        predictions = model.predict(
            X_train.iloc[holdout_idx][ridge_features]
        )

        fold_maes.append(
            mean_absolute_error(
                y_train.iloc[holdout_idx],
                predictions
            )
        )

    final_results.append({
        "alpha": alpha,
        "fold_1_mae": fold_maes[0],
        "fold_2_mae": fold_maes[1],
        "fold_3_mae": fold_maes[2],
        "mean_cv_mae": np.mean(fold_maes),
        "std_cv_mae": np.std(fold_maes)
    })

cv_results_df = (
    pd.concat(
        [cv_results_df, pd.DataFrame(final_results)],
        ignore_index=True
    )
    .drop_duplicates(subset="alpha")
    .sort_values("mean_cv_mae")
    .reset_index(drop=True)
)

display(cv_results_df.round(5))

best_alpha = float(cv_results_df.iloc[0]["alpha"])
print("Selected alpha:", best_alpha)


,alpha,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,std_cv_mae
0,3000.0,0.07256,0.07983,0.07038,0.07425,0.00404
1,1000.0,0.07352,0.07794,0.07261,0.07469,0.00233
2,10000.0,0.07376,0.08355,0.07003,0.07578,0.00570
3,300.0,0.07475,0.07571,0.07756,0.07600,0.00117
4,100.0,0.07552,0.07420,0.08281,0.07751,0.00378
5,10.0,0.07727,0.07601,0.09038,0.08122,0.00650
6,30000.0,0.07923,0.09213,0.07464,0.08200,0.00741
7,1.0,0.07810,0.07782,0.09241,0.08278,0.00681
8,0.1,0.07824,0.07813,0.09269,0.08302,0.00684
9,100000.0,0.09307,0.11033,0.08728,0.09690,0.00979


Selected alpha: 3000.0


In [14]:

# Train Ridge on ALL training fires using selected alpha
final_ridge = clone(ridge_pipeline)
final_ridge.set_params(model__alpha=best_alpha)

final_ridge.fit(
    X_train[ridge_features],
    y_train
)

# Predict for 2018–2019 validation fires
ridge_predictions = final_ridge.predict(
    X_val[ridge_features]
)

# Evaluate using our reusable function
ridge_scores = evaluate_model(
    y_val,
    ridge_predictions,
    val["fire_id"]
)

print(f"RIDGE VALIDATION RESULTS — alpha={best_alpha:g}")
display(pd.Series(ridge_scores).round(5))

# Compare all three prediction approaches
comparison = pd.DataFrame({
    "Training mean": mean_scores,
    "NBR1 persistence": persistence_scores,
    "Ridge regression": ridge_scores,
}).T

print("\nVALIDATION MODEL COMPARISON")
display(comparison.round(5))


RIDGE VALIDATION RESULTS — alpha=3000


mae               0.11308
rmse              0.14032
r2                0.45881
equal_fire_mae    0.11276
dtype: float64


VALIDATION MODEL COMPARISON


,mae,rmse,r2,equal_fire_mae
Training mean,0.16244,0.19316,-0.02555,0.14215
NBR1 persistence,0.18389,0.24230,-0.61385,0.15851
Ridge regression,0.11308,0.14032,0.45881,0.11276


In [15]:

import mlflow

mlflow.set_experiment("P1 - Wildfire Recovery - Phase 5")

with mlflow.start_run(run_name="ridge_alpha_3000") as run:
    mlflow.log_params({
        "model": "Ridge",
        "alpha": best_alpha,
        "dataset": "patch_model_eligible_terrain_v2",
        "n_features": len(ridge_features),
        "excluded_feature": "nbr_drop_patch",
        "cv_method": "GroupKFold",
        "cv_folds": 3,
        "train_fires": 78,
        "validation_fires": 36,
    })

    mlflow.log_metrics({
        "cv_mae": float(cv_results_df.iloc[0]["mean_cv_mae"]),
        "cv_std_mae": float(cv_results_df.iloc[0]["std_cv_mae"]),
        **{
            f"val_{name}": float(value)
            for name, value in ridge_scores.items()
        },
    })

    # Save the fitted preprocessing + Ridge model
    mlflow.sklearn.log_model(
        sk_model=final_ridge,
        name="ridge_pipeline",
    )

print("Ridge experiment recorded!")
print("MLflow run ID:", run.info.run_id)


2026/10/30 21:42:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Ridge experiment recorded!
MLflow run ID: a3d6e79143654e55bfa5e603ffc9e227
